In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "dualtest_experiments" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "DUALTEST"))

print(PROJECT_ROOT)

/home/alumno1/Downloads/NLP_Proyecto_Final-main


In [5]:
import pandas as pd
import numpy as np

from experiment_utils import prepare_records
from prefixing import split_by_tokens
from reference_model import ReferenceModel
from metrics import run_length, edit_similarity, rlb_score, esb_score

In [6]:
reference = ReferenceModel(
    model_name="Qwen/Qwen2.5-0.5B",
    device="cuda"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [7]:
records = prepare_records(
    dataset_name="booktection",
    n=20,
    random_state=7,
    balance_labels=True,
)

len(records), records[0].keys()

(20,
 dict_keys(['id', 'dataset', 'dataset_family', 'source_dataset', 'file_path', 'label', 'estimated_membership', 'text', 'text_hash']))

In [8]:
def score_artificial_controls(records, tokenizer, mode):
    rows = []

    for i, record in enumerate(records):
        split = split_by_tokens(
            record["text"],
            tokenizer=tokenizer,
            prefix_len=64,
            continuation_len=64,
        )

        if mode == "positive":
            # Control positivo: el target copia exactamente la continuación real
            target_text = split.continuation_text
            target_tokens = split.continuation_token_ids

        elif mode == "negative":
            # Control negativo: usa la continuación de otro ejemplo
            other = records[(i + 1) % len(records)]
            other_split = split_by_tokens(
                other["text"],
                tokenizer=tokenizer,
                prefix_len=64,
                continuation_len=64,
            )
            target_text = other_split.continuation_text
            target_tokens = other_split.continuation_token_ids

        else:
            raise ValueError("mode debe ser 'positive' o 'negative'")

        r = run_length(
            target_tokens=target_tokens,
            source_tokens=split.continuation_token_ids,
        )

        s = edit_similarity(
            target_text,
            split.continuation_text,
        )

        r2, p_rlb = rlb_score(
            target_tokens=target_tokens,
            source_tokens=split.continuation_token_ids,
            reference_model=reference,
            prefix_token_ids=split.prefix_token_ids,
        )

        s2, p_esb = esb_score(
            target_text=target_text,
            target_tokens=target_tokens,
            source_text=split.continuation_text,
            reference_model=reference,
            prefix_token_ids=split.prefix_token_ids,
        )

        rows.append({
            "id": record["id"],
            "mode": mode,
            "run_length": r,
            "edit_similarity": s,
            "p_rlb": p_rlb,
            "p_esb": p_esb,
            "prefix": split.prefix_text,
            "ground_truth": split.continuation_text,
            "target_completion": target_text,
        })

    return pd.DataFrame(rows)

In [9]:
df_pos = score_artificial_controls(
    records=records,
    tokenizer=reference.tokenizer,
    mode="positive",
)

df_neg = score_artificial_controls(
    records=records,
    tokenizer=reference.tokenizer,
    mode="negative",
)

df_controls = pd.concat([df_pos, df_neg], ignore_index=True)

df_controls.head()

,id,mode,run_length,edit_similarity,p_rlb,p_esb,prefix,ground_truth,target_completion
0,booktection_15836_A.txt,positive,64,1.0,0.0,0.0,"So much so, that when Camille’s shift came to ...",seemed as if they grew right out of the stone...,seemed as if they grew right out of the stone...
1,booktection_04977_A.txt,positive,64,1.0,0.0,0.0,"""Three's enough."" Piggy's glasses flashed. ""I ...","pool. Piggy hung bumbling behind them. ""If Si...","pool. Piggy hung bumbling behind them. ""If Si..."
2,booktection_11951_A.txt,positive,23,1.0,0.0,0.0,Maybe hes getting lazy. Do you mind if we wait...,? The sheer stupidity of this investigation is...,? The sheer stupidity of this investigation is...
3,booktection_14788_A.txt,positive,64,1.0,0.0,0.0,And while he no doubt advocates more for himse...,be that reckless. It’s something I can’t help...,be that reckless. It’s something I can’t help...
4,booktection_09760_A.txt,positive,64,1.0,0.0,0.0,Christopher Abaddon. He is an invited guest of...,elevator on the right. It goes all the way up...,elevator on the right. It goes all the way up...


In [10]:
df_controls.groupby("mode")[[
    "run_length",
    "edit_similarity",
    "p_rlb",
    "p_esb"
]].mean()

,run_length,edit_similarity,p_rlb,p_esb
mode,,,,
negative,0.00,0.238127,1.000000e+00,9.888291e-40
positive,51.35,1.000000,3.728666e-30,3.728666e-30


In [11]:
df_controls.groupby("mode")[[
    "run_length",
    "edit_similarity",
    "p_rlb",
    "p_esb"
]].median()

,run_length,edit_similarity,p_rlb,p_esb
mode,,,,
negative,0.0,0.236795,1.0,0.0
positive,62.5,1.000000,0.0,0.0


In [ ]:
out_dir = PROJECT_ROOT / "results"
out_dir.mkdir(exist_ok=True)

out_path = out_dir / "artificial_controls_booktection.csv"
df_controls.to_csv(out_path, index=False)

print(out_path)

/home/alumno1/Downloads/NLP_Proyecto_Final-main/results/artificial_controls_booktection.csv


: 